In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

#device = "cuda"


device = "cpu"
#model_path = "ibm-granite/granite-3b-code-base-2k" # pick anyone from above list
model_path = "ibm-granite/granite-4.0-micro"

tokenizer = AutoTokenizer.from_pretrained(model_path)
# drop device_map if running on CPU
model = AutoModelForCausalLM.from_pretrained(model_path, device_map=device)
model.eval()

Loading weights:   0%|          | 0/322 [00:00<?, ?it/s]

GraniteMoeHybridForCausalLM LOAD REPORT from: ibm-granite/granite-4.0-micro
Key                                                         | Status  | 
------------------------------------------------------------+---------+-
model.layers.{0...39}.block_sparse_moe.output_linear.weight | MISSING | 
model.layers.{0...39}.block_sparse_moe.input_linear.weight  | MISSING | 
model.layers.{0...39}.block_sparse_moe.router.layer.weight  | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing form the checkpoint. Consider training on your downstream task.


In [34]:
%%time
# change input text as desired
messages = [
    {"role": "user", "content": "what is the republic"},
 ]
chat = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
# tokenize the text
input_tokens = tokenizer(chat, return_tensors="pt").to(device)
# generate output tokens
output = model.generate(**input_tokens, 
                        max_new_tokens=100)
# decode output tokens into text
output = tokenizer.batch_decode(output)
# print output
print(output[0])

<|start_of_role|>system<|end_of_role|>You are a helpful assistant. Please ensure responses are professional, accurate, and safe.<|end_of_text|>
<|start_of_role|>user<|end_of_role|>what is the republic<|end_of_text|>
<|start_of_role|>assistant<|end_of_role|>Certainly! Here are you! I'm here to assist you with your request. Here are some tips for creating a comprehensive and helpful and professional and always ensure that your request. Here are some tips for creating a comprehensive guide on how to help you with a comprehensive guide on how to help you with a comprehensive guide on how to help you with a comprehensive guide on how to ensure that covers the importance of the importance of the importance of the importance of the importance of the importance of the importance of the importance
CPU times: user 4min 11s, sys: 169 ms, total: 4min 11s
Wall time: 29.8 s


In [10]:
system_prompt="""
* Role: You are a specialized Text Analysis and Summarization Assistant. Your primary function is to help users distill and understand complex or lengthy texts.

* Core Purpose: To extract key information, generate concise summaries, and facilitate deeper comprehension through structured Q&A.

* Primary Goals and Instructions:

  1.Summarization: Upon receiving a text, your first task is to generate a concise, accurate, and well-structured summary. This summary must capture the main themes, key arguments, essential narrative points, and critical conclusions from the source material.

  2.Q&A Generation: Based on the provided text and your own summary, you must be able to derive factual question-and-answer pairs.

    - The questions should be clear and directly related to the important points of the text.

    - The answers must be verbatim or a direct paraphrase of the information contained within the provided text. You must not generate answers from external knowledge or inference not present in the source.

* Fidelity and Accuracy: You are strictly prohibited from adding external information, personal interpretation, or unsupported conclusions. Your responses must be grounded entirely in the text provided by the user.

* Iterative Refinement: You should be prepared to perform these tasks iteratively—for instance, creating a summary first, and then based on that summary and the original text, generating Q&A pairs to test and reinforce understanding.

* Communication Style: Your tone should be professional, helpful, and precise. 

* Expected output is in json 

{
    "text" : {given text},
    "results" : [
        {
            "q" : {question 1},
            "a" : {answer 1}
        },
        {
            "q" : {question 2},
            "a" : {answer 2}
        },
    ]
"""

In [11]:
user_prompt="""
3. There is no difficulty in seeing that Plato’s divisions of knowledge are based, first, on the fundamental antithesis of sensible and intellectual which pervades the whole pre-Socratic philosophy; in which is implied also the opposition of the permanent and the transient, of the universal and the particular. But the age of philosophy in which he lived seemed to require a further distinction; numbers and figures were beginning to separate from ideas. The world could no longer regard justice as a cube, and was learning to see, though imperfectly, that the abstractions of sense were distinct from the abstractions of mind. Between the Eleatic being or essence and the shadows of phenomena, the Pythagorean principle of number found a place, and was, as Aristotle remarks, a conducting medium from one to the other. Hence Plato is led to introduce a third term which had not hitherto entered into the scheme of his philosophy. He had observed the use of mathematics in education; they were the best preparation for higher studies. The subjective relation between them further suggested an objective one; although the passage from one to the other is really imaginary (Metaph.). For metaphysical and moral philosophy has no connexion with mathematics; number and figure are the abstractions of time and space, not the expressions of purely intellectual conceptions. When divested of metaphor, a straight line or a square has no more to do with right and justice than a crooked line with vice. The figurative association was mistaken for a real one; and thus the three latter divisions of the Platonic proportion were constructed.
"""

In [12]:
messages = [
    {"role": "system", "content": f"{system_prompt}"},
    {"role": "user", "content": f"{user_prompt}"}
 ]

In [13]:
messages

[{'role': 'system',
  'content': '\n* Role: You are a specialized Text Analysis and Summarization Assistant. Your primary function is to help users distill and understand complex or lengthy texts.\n\n* Core Purpose: To extract key information, generate concise summaries, and facilitate deeper comprehension through structured Q&A.\n\n* Primary Goals and Instructions:\n\n  1.Summarization: Upon receiving a text, your first task is to generate a concise, accurate, and well-structured summary. This summary must capture the main themes, key arguments, essential narrative points, and critical conclusions from the source material.\n\n  2.Q&A Generation: Based on the provided text and your own summary, you must be able to derive factual question-and-answer pairs.\n\n    - The questions should be clear and directly related to the important points of the text.\n\n    - The answers must be verbatim or a direct paraphrase of the information contained within the provided text. You must not generate

In [17]:
%%time
input_tokens = tokenizer(messages, return_tensors="pt").to(device)
# generate output tokens
output = model.generate(**input_tokens, 
                        max_new_tokens=100)
# decode output tokens into text
output = tokenizer.batch_decode(output)
# print output
print(output[0])

ValueError: text input must be of type `str` (single example), `list[str]` (batch or single pretokenized example) or `list[list[str]]` (batch of pretokenized examples).

In [19]:
input_tokens = tokenizer(messages, return_tensors="pt").to(device)

ValueError: text input must be of type `str` (single example), `list[str]` (batch or single pretokenized example) or `list[list[str]]` (batch of pretokenized examples).

In [20]:
messages

[{'role': 'system',
  'content': '\n* Role: You are a specialized Text Analysis and Summarization Assistant. Your primary function is to help users distill and understand complex or lengthy texts.\n\n* Core Purpose: To extract key information, generate concise summaries, and facilitate deeper comprehension through structured Q&A.\n\n* Primary Goals and Instructions:\n\n  1.Summarization: Upon receiving a text, your first task is to generate a concise, accurate, and well-structured summary. This summary must capture the main themes, key arguments, essential narrative points, and critical conclusions from the source material.\n\n  2.Q&A Generation: Based on the provided text and your own summary, you must be able to derive factual question-and-answer pairs.\n\n    - The questions should be clear and directly related to the important points of the text.\n\n    - The answers must be verbatim or a direct paraphrase of the information contained within the provided text. You must not generate

In [28]:
input_text = system_prompt + \
"""
### text:
""" + user_prompt
input_text

'\n* Role: You are a specialized Text Analysis and Summarization Assistant. Your primary function is to help users distill and understand complex or lengthy texts.\n\n* Core Purpose: To extract key information, generate concise summaries, and facilitate deeper comprehension through structured Q&A.\n\n* Primary Goals and Instructions:\n\n  1.Summarization: Upon receiving a text, your first task is to generate a concise, accurate, and well-structured summary. This summary must capture the main themes, key arguments, essential narrative points, and critical conclusions from the source material.\n\n  2.Q&A Generation: Based on the provided text and your own summary, you must be able to derive factual question-and-answer pairs.\n\n    - The questions should be clear and directly related to the important points of the text.\n\n    - The answers must be verbatim or a direct paraphrase of the information contained within the provided text. You must not generate answers from external knowledge 

In [32]:
input_text="what is the republic?"

In [33]:
# change input text as desired
#input_text = "def generate():"
# tokenize the text
input_tokens = tokenizer(input_text, return_tensors="pt").to(device)

# transfer tokenized inputs to the device
#for i in input_tokens:
#    input_tokens[i] = input_tokens[i].to(device)

# generate output tokens
output = model.generate(**input_tokens)
# decode output tokens into text
output = tokenizer.batch_decode(output)

# loop over the batch to print, in this example the batch size is 1
for i in output:
    print(i)

what is the republic? a republicanism and what is it's role in the role in the role in the role of the
